In [79]:
!pip install rdflib
!pip install pandas
!pip install pydicom


[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


In [80]:
import os, pickle, joblib

# Get the directory of the current script
base_dir = os.getcwd()
try:
    if folder:
        base_dir = os.path.join(base_dir, folder)
except:
    pass

In [ ]:
from rdflib import * 
import uuid
from hashlib import sha256
import pandas as pd
import pydicom

In [ ]:
tbox = Namespace('http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#')
abox = Namespace('http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#')

# W3C Standard Vocabularies
dcat = Namespace('http://www.w3.org/ns/dcat#')
csvw = Namespace('http://www.w3.org/ns/csvw#')
dcterms = Namespace('http://purl.org/dc/terms/')
schema = Namespace('http://schema.org/')
prov = Namespace('http://www.w3.org/ns/prov#')
xsd = Namespace('http://www.w3.org/2001/XMLSchema#')

The Profiler is a process which simulates a semi-automatic tools to extract metadata from different data assets. Thus, for tabular data assets the schema is extracted and for Images in DICOM, we extract Header Metadata.

In [ ]:
class Profiler:
    """
    Profiler class is responsible for extracting metadata from data files.
    Generates standards-compliant RDF using DCAT, CSVW, Schema.org, and PROV-O.
    """

    def __init__(self, file_path, owner="Unknown"):
        # Access global namespaces
        global tbox, abox, dcat, csvw, dcterms, schema, prov, xsd
        
        self.file_path = file_path
        self.file_extension = os.path.splitext(file_path)[1]
        self.filename = os.path.basename(file_path).split(self.file_extension)[0]
        self.datasetname = self.filename.replace('.', '').replace('-', '_')
        self.owner = owner
        
        # Initialize RDF graph
        self.source_graph = Graph()
        self.source_graph.bind('tb', tbox)
        self.source_graph.bind('ab', abox)
        self.source_graph.bind('dcat', dcat)
        self.source_graph.bind('csvw', csvw)
        self.source_graph.bind('dcterms', dcterms)
        self.source_graph.bind('schema', schema)
        self.source_graph.bind('prov', prov)
        self.source_graph.bind('xsd', xsd)
        
        # Extract metadata
        self.extract_metadata()

    def generate_hash(self):
        """Generate SHA256 hash of the file."""
        hash_algo = sha256()
        with open(self.file_path, 'rb') as f:
            for chunk in iter(lambda: f.read(4096), b""):
                hash_algo.update(chunk)
        return hash_algo.hexdigest()

    def extract_metadata(self):
        """Extract metadata based on file type."""
        global tbox, abox, dcat, csvw, dcterms, schema, prov, xsd
        
        # Create dataset URI
        dataset_uri = abox[self.datasetname]
        
        # Type declarations - both custom and standard
        self.source_graph.add((dataset_uri, RDF.type, tbox.DataProduct))
        self.source_graph.add((dataset_uri, RDF.type, dcat.Dataset))
        
        # Basic metadata
        file_hash = self.generate_hash()
        self.source_graph.add((dataset_uri, dcterms.identifier, Literal(file_hash)))
        self.source_graph.add((dataset_uri, dcterms.title, Literal(f"{self.filename} Dataset", lang="en")))
        self.source_graph.add((dataset_uri, dcterms.creator, Literal(self.owner, lang="en")))
        self.source_graph.add((dataset_uri, tbox.owner, Literal(self.owner)))
        
        # Technology Aspects -> DCAT Distribution
        ta_uri = abox[f'{self.datasetname}_TA']
        self.source_graph.add((ta_uri, RDF.type, tbox.TechnologyAspects))
        self.source_graph.add((ta_uri, RDF.type, dcat.Distribution))
        self.source_graph.add((dataset_uri, tbox.hasTA, ta_uri))
        self.source_graph.add((dataset_uri, dcat.distribution, ta_uri))
        
        # Format and media type
        if self.file_extension.lower() == '.csv':
            self.source_graph.add((dataset_uri, tbox.hasDTT, abox.Tabular))
            self.source_graph.add((ta_uri, dcterms['format'], Literal("CSV")))
            self.source_graph.add((ta_uri, dcat.mediaType, Literal("text/csv")))
            self.extract_csv_metadata()
        elif self.file_extension.lower() == '.dcm':
            self.source_graph.add((dataset_uri, tbox.hasDTT, abox.Image))
            self.source_graph.add((ta_uri, dcterms['format'], Literal("DICOM")))
            self.source_graph.add((ta_uri, dcat.mediaType, Literal("application/dicom")))
            self.extract_dicom_metadata()
        elif self.file_extension.lower() == '.pkl':
            self.source_graph.add((dataset_uri, tbox.hasDTT, abox.Model))
            self.source_graph.add((ta_uri, dcterms['format'], Literal("PKL")))
            self.source_graph.add((ta_uri, dcat.mediaType, Literal("application/octet-stream")))
            test_csv = os.path.join(os.path.dirname(self.file_path), "test.csv")
            if os.path.exists(test_csv):
                self.extract_csv_metadata(test_csv)
        else:
            raise ValueError(f"Unsupported file extension: {self.file_extension}")
        
        # Access information
        acces_uri = abox[f'{self.datasetname}_Acces']
        self.source_graph.add((ta_uri, tbox.hasAcces, acces_uri))
        self.source_graph.add((acces_uri, RDF.type, tbox.Acces))
        self.source_graph.add((acces_uri, RDFS.label, abox.Static))
        self.source_graph.add((acces_uri, tbox.path, Literal(self.file_path)))
        self.source_graph.add((ta_uri, dcat.accessURL, Literal(f"file://{self.file_path}", datatype=XSD.anyURI)))
        
        return self.source_graph

    def extract_csv_metadata(self, csv_file=None):
        """Extract metadata from CSV file and create CSVW-compliant attributes."""
        global tbox, abox, csvw
        
        file_to_read = csv_file if csv_file else self.file_path
        df = pd.read_csv(file_to_read)
        
        dataset_uri = abox[self.datasetname]
        
        for column in df.columns:
            # Create attribute URI
            attr_uri = abox[column]
            
            # Type declarations - both custom and standard
            self.source_graph.add((attr_uri, RDF.type, tbox.Attribute))
            self.source_graph.add((attr_uri, RDF.type, csvw.Column))
            
            # Link to dataset
            self.source_graph.add((dataset_uri, tbox.hasAttribute, attr_uri))
            self.source_graph.add((dataset_uri, csvw.column, attr_uri))
            
            # Column name (both custom and standard)
            self.source_graph.add((attr_uri, tbox.attribute, Literal(column)))
            self.source_graph.add((attr_uri, csvw.name, Literal(column)))
            
            # Add basic datatype inference
            dtype = str(df[column].dtype)
            if 'int' in dtype:
                self.source_graph.add((attr_uri, csvw.datatype, Literal("integer")))
            elif 'float' in dtype:
                self.source_graph.add((attr_uri, csvw.datatype, Literal("number")))
            else:
                self.source_graph.add((attr_uri, csvw.datatype, Literal("string")))
        
        return self.source_graph

    def extract_dicom_metadata(self, n_attributes=50):
        """Extract metadata from DICOM file."""
        global tbox, abox, csvw
        
        ds = pydicom.dcmread(self.file_path)
        dataset_uri = abox[self.datasetname]
        
        # Iterate over DICOM attributes
        for attribute in dir(ds)[:n_attributes]:
            if attribute[0].isalpha() and hasattr(ds, attribute):
                attr_uri = abox[attribute]
                
                # Type declarations
                self.source_graph.add((attr_uri, RDF.type, tbox.Attribute))
                self.source_graph.add((attr_uri, RDF.type, csvw.Column))
                
                # Link to dataset
                self.source_graph.add((dataset_uri, tbox.hasAttribute, attr_uri))
                self.source_graph.add((dataset_uri, csvw.column, attr_uri))
                
                # Attribute name
                self.source_graph.add((attr_uri, tbox.attribute, Literal(attribute)))
                self.source_graph.add((attr_uri, csvw.name, Literal(attribute)))
        
        return self.source_graph

    def get_source_graph(self):
        return self.source_graph

In [83]:
#file_path = "/home/acraf/psr/Fdatavalidation/DataProductLayer/DataProduct3/Data/Explotation/SurvivalClassifier.pkl"

In [ ]:
try:
    if file_path:
        # If file_path is absolute, use it as-is; otherwise join with base_dir
        if not os.path.isabs(file_path):
            file_path = os.path.join(base_dir, file_path)
except:
    file_path = input("Enter file path: ")

In [85]:
profiler = Profiler(file_path)

http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#MGMT_Not Available does not look like a valid URI, trying to serialize this will break.
http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#MGMT_Not Available does not look like a valid URI, trying to serialize this will break.
http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#MGMT_Not Available does not look like a valid URI, trying to serialize this will break.
http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#GTR_over90percent_Not Applicable does not look like a valid URI, trying to serialize this will break.
http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#GTR_over90percent_Not Applicable does not look like a valid URI, trying to serialize this will break.
http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#GTR_over90percent_Not Applicable does not look like a valid URI, trying to serialize this will break.
http://www.semanticweb.org/acraf/ontolog

OTHERFIle /home/acraf/psr/Fdatavalidation/DataProductLayer/DataProduct3/Data/Explotation/X_test.csv


In [86]:
graph = profiler.get_source_graph()

Save Graph to File

In [31]:
sdm = Graph().parse(os.path.join(base_dir, '../../FederatedComputationalGovernance/SemanticDataModel/sdm.ttl'), format='turtle')

In [32]:
sdm = Graph().parse(os.path.join(base_dir, '../../FederatedComputationalGovernance/SemanticDataModel/sdm.ttl'), format='turtle')
sdm += graph
sdm.serialize(destination=os.path.join(base_dir, '../../FederatedComputationalGovernance/SemanticDataModel/sdm.ttl'), format='turtle')

TypeError: 'NoneType' object is not iterable